# TUGAS 02 - EKSTRAKSI INFORMASI
## Uji Coba Rule-Based Information Extraction Sederhana

**Identitas Mahasiswa:**
- **NIM:** 240411100135
- **Mata Kuliah:** Ekstraksi Informasi
- **Objek Kajian:** Dokumen Putusan Pengadilan Negeri Blitar (PN Blitar)

---
### Deskripsi Tugas:
1. Melakukan konversi dokumen putusan pengadilan format **PDF** hasil scraping tugas sebelumnya menjadi format teks (**TXT**).
2. Membangun dan menguji sistem ekstraksi informasi berbasis aturan (**Rule-Based Information Extraction**) menggunakan Python dan Regular Expression (`re`).
3. Mengekstrak entitas-entitas penting dari minimal 5 dokumen putusan perkara pidana:
   - **Nomor Putusan**
   - **Nama Terdakwa** (Terdakwa 1, 2, 3)
   - **Tuntutan Jaksa Penuntut Umum** (Tindak Pidana, Pasal KUHP/UU yang dilanggar, Lama Tuntutan Hukuman)
   - **Putusan / Vonis Majelis Hakim** (Tindak Pidana Terbukti, Amar Putusan Hukuman Penjara/Denda)
   - **Aparatur Persidangan** (Tanggal Putusan, Hakim Ketua, Hakim Anggota 1, Hakim Anggota 2, Panitera Pengganti, Jaksa Penuntut Umum)
4. Menyimpan output ekstraksi ke format teks individual (`OUTPUT/O-[namafile].txt`) dan format tabel (`courtHistory.csv`).

## 1. Import Library yang Dibutuhkan

Tahap pertama adalah memuat seluruh pustaka Python yang diperlukan:
- `fitz` (`PyMuPDF`): Digunakan untuk membaca dan mengekstrak teks dari dokumen PDF putusan pengadilan secara cepat dan akurat.
- `os`: Digunakan untuk manipulasi path folder input/output dan membaca daftar file.
- `re`: Modul Regular Expression sebagai mesin utama dalam pencocokan pola aturan teks (*rule-based pattern matching*).
- `csv`: Modul untuk membaca dan menulis hasil ekstraksi ke file CSV berbentuk tabel.
- `pandas`: Digunakan untuk menyajikan tabel hasil ekstraksi secara rapi dan terstruktur.

In [1]:
# Import modul standar Python untuk manajemen file dan sistem operasi
import os

# Import pustaka Regular Expression untuk pencocokan pola aturan teks (Rule-Based IE)
import re

# Import pustaka CSV untuk mengekspor data tabel hasil ekstraksi
import csv

# Import PyMuPDF (fitz) untuk ekstraksi teks dari dokumen PDF
import fitz

# Import pandas untuk visualisasi dan verifikasi data hasil ekstraksi
import pandas as pd

print("Seluruh library berhasil dimuat dengan sukses!")

Seluruh library berhasil dimuat dengan sukses!


## 2. Konversi Dokumen Putusan dari PDF ke Teks (TXT)

Sesuai arahan tugas, dokumen putusan pengadilan yang sebelumnya diunduh dalam bentuk file `.pdf` dari Direktori Putusan Mahkamah Agung (PN Blitar) dikonversi menjadi file teks mentah `.txt`.

Fungsi `convert_pdf_to_txt` membaca seluruh halaman dokumen PDF halaman per halaman, mengambil konten teksnya, lalu menyimpannya ke folder `./INPUT_TXT/`.

In [2]:
def convert_pdf_to_txt(pdf_path, txt_path):
    """
    Fungsi untuk mengonversi satu file PDF menjadi file teks mentah (.txt).
    Parameter:
      - pdf_path : Path lengkap file PDF asal
      - txt_path : Path tujuan penyimpanan file TXT
    """
    # Buka dokumen PDF menggunakan engine PyMuPDF
    doc = fitz.open(pdf_path)
    full_text = []
    
    # Ekstraksi teks dari setiap halaman PDF secara berurutan
    for page_num in range(len(doc)):
        page = doc[page_num]
        full_text.append(page.get_text())
    doc.close()
    
    # Simpan seluruh teks yang tergabung ke file TXT dengan encoding UTF-8
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write('\n'.join(full_text))
        
    return txt_path

# Inisialisasi direktori folder input PDF dan output teks TXT
folder_pdf = "./INPUT_PDF/"
folder_txt = "./INPUT_TXT/"
os.makedirs(folder_txt, exist_ok=True)

# Ambil seluruh file PDF yang ada pada folder INPUT_PDF
daftar_file_pdf = sorted([f for f in os.listdir(folder_pdf) if f.endswith('.pdf')])

print(f"Memulai proses konversi {len(daftar_file_pdf)} dokumen PDF ke TXT...\n")
for file_pdf in daftar_file_pdf:
    path_asal = os.path.join(folder_pdf, file_pdf)
    nama_txt = file_pdf.replace('.pdf', '.txt')
    path_tujuan = os.path.join(folder_txt, nama_txt)
    
    convert_pdf_to_txt(path_asal, path_tujuan)
    ukuran_kb = os.path.getsize(path_tujuan) / 1024
    print(f"[Berhasil Konversi] {file_pdf} -> {nama_txt} ({ukuran_kb:.2f} KB)")

print("\nSeluruh dokumen putusan berhasil dikonversi ke format teks TXT!")

Memulai proses konversi 5 dokumen PDF ke TXT...

[Berhasil Konversi] putusan_207_pid.sus_2022_pn_blt_20261001203703.pdf -> putusan_207_pid.sus_2022_pn_blt_20261001203703.txt (75.46 KB)
[Berhasil Konversi] putusan_313_pid.c_2020_pn_blt_20261001203239.pdf -> putusan_313_pid.c_2020_pn_blt_20261001203239.txt (5.78 KB)
[Berhasil Konversi] putusan_32_pid.b_2026_pn_blt_20261001202728.pdf -> putusan_32_pid.b_2026_pn_blt_20261001202728.txt (44.51 KB)
[Berhasil Konversi] putusan_34_pid.c_2026_pn_blt_20261001015134.pdf -> putusan_34_pid.c_2026_pn_blt_20261001015134.txt (18.71 KB)
[Berhasil Konversi] putusan_62_pid.b_2026_pn_blt_20261001014143.pdf -> putusan_62_pid.b_2026_pn_blt_20261001014143.txt (68.28 KB)

Seluruh dokumen putusan berhasil dikonversi ke format teks TXT!


## 3. Implementasi Mesin Rule-Based Information Extraction (Regex)

Di bawah ini adalah fungsi inti `generateEntity(pathFile, fileName)` yang mengimplementasikan aturan dan pola Regular Expression (`re`) untuk mengekstrak entitas-entitas perkara dari teks putusan pengadilan.

### Aturan Pola yang Diterapkan:
1. **Nomor Putusan**: Mencari pola standar penomoran registrasi perkara pengadilan, contoh: `62/Pid.B/2026/PN Blt`.
2. **Identitas Terdakwa**: Mengekstrak nama terdakwa dari bagian tabel identitas perkara (`Nama lengkap : [nama]`) maupun dakwaan alternatif.
3. **Tuntutan Jaksa Penuntut Umum**: Mengekstrak pasal KUHP/UU yang dituntutkan, jenis tindak pidana tuntutan, dan lama pidana penjara yang diminta jaksa.
4. **Putusan / Vonis Majelis Hakim**: Mengekstrak amar putusan pada bagian `MENGADILI`, mencakup tindak pidana yang terbukti secara sah dan meyakinkan, serta amar hukuman (penjara/kurungan/denda).
5. **Aparatur Persidangan**: Mengekstrak tanggal pembacaan putusan, Hakim Ketua, Hakim Anggota 1 & 2, Panitera Pengganti, serta Jaksa Penuntut Umum pada bagian penutup (*Demikianlah diputuskan...*).

In [3]:
def clean_whitespace(text):
    """Membersihkan karakter spasi ganda, enter ganda, dan tabulasi liar."""
    if not text:
        return ""
    text = re.sub(r'[\r\n\t]+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def generateEntity(folderData, fileName):
    """
    Fungsi Rule-Based IE untuk mengekstrak 15 atribut entitas perkara dari teks putusan.
    Output berupa dictionary berisi seluruh entitas yang diekstrak.
    """
    path_file = os.path.join(folderData, fileName)
    with open(path_file, 'r', encoding='utf-8', errors='ignore') as f:
        text = f.read()

    hasil_entitas = {}
    
    # =========================================================================
    # 1. ENTITAS: NOMOR PUTUSAN
    # Pola: [nomor urut]/[kode perkara: Pid.B / Pid.Sus / Pid.C]/[tahun]/PN Blt
    # =========================================================================
    m_nomor = re.search(r'([0-9]+\s*/\s*[A-Za-z0-9\.\-]+\s*/\s*[0-9]{4}\s*/\s*PN(?:\.|\s*)Blt)', text, re.I)
    if not m_nomor:
        m_nomor = re.search(r'Nomor\s*[:\.]?\s*([0-9]+/[A-Za-z0-9\.\-]+/[0-9]{4}/[A-Za-z0-9\.\s]+)', text, re.I)
    hasil_entitas['Nomor Putusan'] = clean_whitespace(m_nomor.group(1)) if m_nomor else ""

    # =========================================================================
    # 2. ENTITAS: IDENTITAS TERDAKWA (Terdakwa 1, Terdakwa 2, Terdakwa 3)
    # Pola: Bagian tabel identitas perkara "Nama lengkap : [Nama Terdakwa]"
    # =========================================================================
    list_terdakwa = []
    m_terdakwa = re.findall(r'(?:Nama\s+lengkap|Nama\s+terdakwa|Nama\s+Lengkap)\s*[:\.]?\s*([^\r\n;]+)', text, re.I)
    for t in m_terdakwa:
        nama_bersih = clean_whitespace(t).rstrip(';.,')
        if nama_bersih and len(nama_bersih) > 2 and nama_bersih.lower() not in [x.lower() for x in list_terdakwa]:
            list_terdakwa.append(nama_bersih)
            
    # Pola cadangan jika tidak menggunakan format tabel standar
    if not list_terdakwa:
        m_alt = re.findall(r'perkara\s+Terdakwa\s*[:\.]?\s*(?:1\.\s*)?([A-Z\s\.,]{3,50})(?:bin|binti|lahir|,|\n)', text, re.I)
        for t in m_alt:
            ct = clean_whitespace(t).rstrip(';.,')
            if ct and ct.lower() not in [x.lower() for x in list_terdakwa]:
                list_terdakwa.append(ct)
                
    hasil_entitas['Nama Terdakwa 1'] = list_terdakwa[0] if len(list_terdakwa) > 0 else ""
    hasil_entitas['Nama Terdakwa 2'] = list_terdakwa[1] if len(list_terdakwa) > 1 else ""
    hasil_entitas['Nama Terdakwa 3'] = list_terdakwa[2] if len(list_terdakwa) > 2 else ""

    # =========================================================================
    # 3. ENTITAS: TUNTUTAN JAKSA PENUNTUT UMUM
    # Mengambil bagian isi surat tuntutan pidana yang diajukan Penuntut Umum
    # =========================================================================
    hasil_entitas['Tuntutan Pidana'] = ""
    hasil_entitas['Tuntutan KUHP'] = ""
    hasil_entitas['Tuntukan Hukuman'] = ""
    
    m_tuntutan_sec = re.search(r'(?:tuntutan\s+pidana|menuntut\s+supaya|surat\s+tuntutan)(.*?)(?:M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I|Menimbang|MENGADILI)', text, re.DOTALL | re.I)
    sec_tuntutan = m_tuntutan_sec.group(1) if m_tuntutan_sec else text
    
    # a. Tindak Pidana Tuntutan
    m_tp_tuntutan = re.search(r'bersalah\s+melakukan\s+tindak\s+pidana\s+["“\']?([^"”\';\n\r]+)["”\']?', sec_tuntutan, re.I)
    if m_tp_tuntutan:
        hasil_entitas['Tuntutan Pidana'] = clean_whitespace(m_tp_tuntutan.group(1)).strip(';.,')
        
    # b. Pasal KUHP / Undang-Undang yang Dituntutkan
    m_kuhp_tuntutan = re.search(r'((?:Pasal\s+[\d\w\s\(\)]+)\s+(?:KUHP|KUHPidana|UU\s+[\d\w\s/]+|UURI\s+[\d\w\s/]+))', sec_tuntutan, re.I)
    if m_kuhp_tuntutan:
        hasil_entitas['Tuntutan KUHP'] = clean_whitespace(m_kuhp_tuntutan.group(1))

    # c. Lama Tuntutan Hukuman Penjara
    m_huk_tuntutan = re.search(r'(?:pidana\s+penjara\s+selama|penjara\s+selama|berupa\s+pidana\s+penjara\s+selama)\s+([0-9\(\)\w\s]+?)(?:dikurangi|dengan|denda|menetapkan|;|\.|$)', sec_tuntutan, re.I)
    if m_huk_tuntutan:
        hasil_entitas['Tuntukan Hukuman'] = clean_whitespace(m_huk_tuntutan.group(1)).strip(';.,')

    # =========================================================================
    # 4. ENTITAS: PUTUSAN / VONIS MAJELIS HAKIM (AMAR PUTUSAN)
    # Mengambil bagian amar putusan setelah kata 'M E N G A D I L I :'
    # =========================================================================
    hasil_entitas['Putusan Pidana'] = ""
    hasil_entitas['Putusan Hukuman'] = ""
    
    m_mengadili_sec = re.search(r'(?:M\s*E\s*N\s*G\s*A\s*D\s*I\s*L\s*I|MENGADILI)\s*:(.*?)(?:Demikianlah\s+diputuskan|Demikian\s+diputuskan|$)', text, re.DOTALL | re.I)
    sec_putusan = m_mengadili_sec.group(1) if m_mengadili_sec else text
    
    # a. Tindak Pidana Putusan yang Terbukti
    m_tp_putusan = re.search(r'(?:bersalah\s+melakukan\s+tindak\s+pidana|melakukan\s+tindak\s+pidana)\s+["“\']?([^"”\';\n\r]+)["”\']?', sec_putusan, re.I)
    if m_tp_putusan:
        hasil_entitas['Putusan Pidana'] = clean_whitespace(m_tp_putusan.group(1)).strip(';.,?')
        
    # b. Amar Vonis Hukuman (Penjara, Kurungan, atau Denda)
    m_huk_putusan = re.search(r'(?:pidana\s+penjara\s+selama|pidana\s+kurungan\s+selama|penjara\s+selama|kurungan\s+selama|pidana\s+denda\s+sejumlah|denda\s+sejumlah)\s+([^;\.\n\r]+)', sec_putusan, re.I)
    if m_huk_putusan:
        hasil_entitas['Putusan Hukuman'] = clean_whitespace(m_huk_putusan.group(1)).strip(';.,')

    # =========================================================================
    # 5. ENTITAS: APARATUR PERSIDANGAN (Tanggal, Hakim, Panitera, Penuntut)
    # Mengambil bagian musyawarah penutup pengadilan
    # =========================================================================
    hasil_entitas['Tanggal Putusan'] = ""
    hasil_entitas['Hakim Ketua'] = ""
    hasil_entitas['Hakim Anggota1'] = ""
    hasil_entitas['Hakim Anggota2'] = ""
    hasil_entitas['Panitera'] = ""
    hasil_entitas['Penuntut Umum'] = ""
    
    m_penutup = re.search(r'(?:Demikianlah\s+diputuskan|Demikian\s+diputuskan)(.*)', text, re.DOTALL | re.I)
    sec_penutup = m_penutup.group(1) if m_penutup else text[-1800:]
    sec_penutup_clean = clean_whitespace(sec_penutup)
    
    # a. Tanggal Pembacaan Putusan
    m_tgl = re.search(r'tanggal\s+([0-9]+\s+[A-Za-z]+\s+[0-9]{4})', sec_penutup_clean, re.I)
    if m_tgl:
        hasil_entitas['Tanggal Putusan'] = m_tgl.group(1)
        
    # b. Hakim Ketua (Majelis) atau Hakim Tunggal
    m_hk = re.search(r'oleh\s+(?:kami,\s*)?([A-Za-z\s\.,]+?)\s*,\s*sebagai\s+Hakim\s+Ketua', sec_penutup_clean, re.I)
    if not m_hk:
        # Format persidangan catatan cepat (Tipiring/Hakim Tunggal)
        m_hk = re.search(r'([A-Z\s\.,]{5,50})\s*:\s*Hakim;', text, re.I)
    if not m_hk:
        m_hk = re.search(r'oleh\s+([A-Za-z\s\.,]+?)\s*,\s*S?\.?H?\.?\s*(?:selaku\s+)?Hakim', sec_penutup_clean, re.I)
    if not m_hk:
        m_hk = re.search(r'Hakim\s*,\s*([A-Za-z\s\.,]+?)(?:Panitera|$)', sec_penutup_clean, re.I)
    hasil_entitas['Hakim Ketua'] = clean_whitespace(m_hk.group(1)).strip(';.,') if m_hk else ""

    # c. Hakim Anggota 1 & Hakim Anggota 2
    m_ha = re.search(r'Hakim\s+Ketua\s*,\s*(.*?)\s*(?:masing\s*-\s*masing|dan)\s+sebagai\s+Hakim\s+Anggota', sec_penutup_clean, re.I)
    if m_ha:
        ha_text = m_ha.group(1)
        parts = re.split(r'\s*,\s*(?=[A-Z][a-z]+(?:\s+[A-Z][a-z]+)*\s*,\s*S\.H)', ha_text)
        if len(parts) >= 2:
            hasil_entitas['Hakim Anggota1'] = clean_whitespace(parts[0]).strip(';.,')
            hasil_entitas['Hakim Anggota2'] = clean_whitespace(parts[1]).strip(';.,')
        else:
            parts = re.split(r'\s+dan\s+|\s*,\s*(?=[A-Z])', ha_text)
            if len(parts) >= 2:
                hasil_entitas['Hakim Anggota1'] = clean_whitespace(parts[0]).strip(';.,')
                hasil_entitas['Hakim Anggota2'] = clean_whitespace(parts[1]).strip(';.,')
            else:
                hasil_entitas['Hakim Anggota1'] = clean_whitespace(ha_text).strip(';.,')

    # d. Panitera Pengganti
    m_panitera = re.search(r'dibantu\s+oleh\s+([A-Za-z\s\.,]+?)\s*,\s*(?:selaku\s+)?Panitera\s+Pengganti', sec_penutup_clean, re.I)
    if not m_panitera:
        m_panitera = re.search(r'([A-Z\s\.,]{5,50})\s*:\s*Panitera\s+Pengganti;', text, re.I)
    if not m_panitera:
        m_panitera = re.search(r'Panitera\s+Pengganti\s*[:\.]?\s*([A-Za-z\s\.,]+?)(?:Hakim|pada|\n|$)', sec_penutup_clean, re.I)
    hasil_entitas['Panitera'] = clean_whitespace(m_panitera.group(1)).strip(';.,') if m_panitera else ""

    # e. Jaksa Penuntut Umum
    m_pu = re.search(r'(?:dihadiri|dihadiri\s+pula)\s+oleh\s+([A-Za-z\s\.,]+?)\s*,\s*(?:selaku\s+)?(?:Jaksa\s+)?Penuntut\s+Umum', sec_penutup_clean, re.I)
    if not m_pu:
        m_pu = re.search(r'Penuntut\s+Umum\s*[:\.]?\s*([A-Za-z\s\.,]+?)(?:dan|pada|\n|$)', sec_penutup_clean, re.I)
    hasil_entitas['Penuntut Umum'] = clean_whitespace(m_pu.group(1)).strip(';.,') if m_pu else ""

    return hasil_entitas

print("Fungsi generateEntity berhasil didefinisikan dan siap digunakan!")

Fungsi generateEntity berhasil didefinisikan dan siap digunakan!


## 4. Fungsi Penyimpanan Output (Individual TXT & CSV Terstruktur)

Dua fungsi pembantu disiapkan:
1. `save_individual_output`: Menyimpan hasil ekstraksi entitas setiap dokumen putusan ke dalam file teks individual di folder `./OUTPUT/O-[namafile].txt`.
2. `save_to_csv`: Menyimpan seluruh kumpulan entitas putusan ke dalam file tabel `courtHistory.csv` sesuai format yang diinstruksikan oleh dosen.

In [4]:
def save_individual_output(entity_dict, output_folder, file_name):
    """
    Menyimpan hasil ekstraksi entitas untuk satu file putusan ke file teks individual O-[namafile].txt
    """
    os.makedirs(output_folder, exist_ok=True)
    nama_file_output = "O-" + file_name
    path_output = os.path.join(output_folder, nama_file_output)
    
    with open(path_output, 'w', encoding='utf-8') as f:
        f.write("=" * 65 + "\n")
        f.write("HASIL EKSTRAKSI INFORMASI RULE-BASED (PENGADILAN NEGERI BLITAR)\n")
        f.write(f"Dokumen Asal: {file_name}\n")
        f.write("=" * 65 + "\n\n")
        
        f.write(f"Nomor Putusan   : {entity_dict.get('Nomor Putusan', '')}\n\n")
        
        f.write("--- IDENTITAS TERDAKWA ---\n")
        f.write(f"Nama Terdakwa 1 : {entity_dict.get('Nama Terdakwa 1', '')}\n")
        f.write(f"Nama Terdakwa 2 : {entity_dict.get('Nama Terdakwa 2', '')}\n")
        f.write(f"Nama Terdakwa 3 : {entity_dict.get('Nama Terdakwa 3', '')}\n\n")
        
        f.write("--- TUNTUTAN JAKSA PENUNTUT UMUM ---\n")
        f.write(f"Tindak Pidana   : {entity_dict.get('Tuntutan Pidana', '')}\n")
        f.write(f"Melanggar Pasal : {entity_dict.get('Tuntutan KUHP', '')}\n")
        f.write(f"Tuntutan Hukuman: {entity_dict.get('Tuntukan Hukuman', '')}\n\n")
        
        f.write("--- PUTUSAN / VONIS MAJELIS HAKIM ---\n")
        f.write(f"Tindak Pidana   : {entity_dict.get('Putusan Pidana', '')}\n")
        f.write(f"Amar Putusan    : {entity_dict.get('Putusan Hukuman', '')}\n\n")
        
        f.write("--- APARATUR PERSIDANGAN ---\n")
        f.write(f"Tanggal Putusan : {entity_dict.get('Tanggal Putusan', '')}\n")
        f.write(f"Hakim Ketua     : {entity_dict.get('Hakim Ketua', '')}\n")
        f.write(f"Hakim Anggota 1 : {entity_dict.get('Hakim Anggota1', '')}\n")
        f.write(f"Hakim Anggota 2 : {entity_dict.get('Hakim Anggota2', '')}\n")
        f.write(f"Panitera        : {entity_dict.get('Panitera', '')}\n")
        f.write(f"Penuntut Umum   : {entity_dict.get('Penuntut Umum', '')}\n")
        f.write("=" * 65 + "\n")
        
    return path_output

def save_to_csv(list_entities, csv_path):
    """
    Menyimpan list entitas seluruh putusan ke file CSV berformat tabel.
    """
    header_kolom = [
        'Nomor Putusan', 'Nama Terdakwa 1', 'Nama Terdakwa 2', 'Nama Terdakwa 3',
        'Tuntutan Pidana', 'Tuntutan KUHP', 'Tuntukan Hukuman', 'Putusan Pidana',
        'Putusan Hukuman', 'Tanggal Putusan', 'Hakim Ketua', 'Hakim Anggota1',
        'Hakim Anggota2', 'Panitera', 'Penuntut Umum'
    ]
    
    with open(csv_path, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=header_kolom)
        writer.writeheader()
        for data in list_entities:
            writer.writerow(data)
            
    return csv_path

print("Fungsi penyimpanan output teks dan CSV berhasil dibuat!")

Fungsi penyimpanan output teks dan CSV berhasil dibuat!


## 5. Eksekusi Ekstraksi pada 5 Dokumen Putusan PN Blitar

Pada tahap ini, script membaca seluruh file dokumen teks yang telah dikonversi di `./INPUT_TXT/`, menjalankan fungsi `generateEntity()`, mencetak hasil ekstraksi ke layar, dan menyimpan hasilnya ke file teks perorangan di `./OUTPUT/` serta ke file CSV `courtHistory.csv`.

In [5]:
# Konfigurasi folder dan file output
start_path = "./INPUT_TXT/"
hasil_path = "./OUTPUT/"
csv_output = "courtHistory.csv"

# Ambil daftar file teks hasil konversi PDF putusan PN Blitar
daftar_file = sorted([f for f in os.listdir(start_path) if f.endswith('.txt')])
list_semua_hasil = []

print(f"Memulai proses ekstraksi informasi dari {len(daftar_file)} dokumen putusan...\n")

for idx, tiap_file in enumerate(daftar_file, 1):
    print(f"=================================================================")
    print(f"DOKUMEN [{idx}/{len(daftar_file)}]: {tiap_file}")
    print(f"=================================================================")
    
    # Ekstraksi entitas menggunakan rule-based engine
    hasil_ekstraksi = generateEntity(start_path, tiap_file)
    list_semua_hasil.append(hasil_ekstraksi)
    
    # Tampilkan atribut entitas yang berhasil diperoleh
    for atribut, nilai in hasil_ekstraksi.items():
        print(f"{atribut:18} : {nilai}")
        
    # Simpan hasil ekstraksi individu ke folder OUTPUT/O-[namafile].txt
    file_simpan = save_individual_output(hasil_ekstraksi, hasil_path, tiap_file)
    print(f"-> File tersimpan: {file_simpan}\n")

# Simpan seluruh data entitas ke file CSV tabel
save_to_csv(list_semua_hasil, csv_output)
print("=================================================================")
print(f"PROSES SELESAI! Seluruh entitas berhasil diekspor ke: {csv_output}")
print("=================================================================")

Memulai proses ekstraksi informasi dari 5 dokumen putusan...



DOKUMEN [1/5]: putusan_207_pid.sus_2022_pn_blt_20261001203703.txt
Nomor Putusan      : 207/Pid.Sus/2022/PN Blt
Nama Terdakwa 1    : Kateni Bin Alm Jaimin
Nama Terdakwa 2    : 
Nama Terdakwa 3    : 
Tuntutan Pidana    : menganiaya dan/atau
Tuntutan KUHP      : Pasal 91 B ayat (1) jo Pasal 66 A ayat (1) UU RI No
Tuntukan Hukuman   : 4 (empat) Bulan dan
Putusan Pidana     : PENGANIAYAAN
Putusan Hukuman    : 4 (empat) Bulan dan denda sebesar Rp 3
Tanggal Putusan    : 29 Agustus 2022
Hakim Ketua        : Dr. Wisnu Widiastuti, S.H.,, M.H.um
Hakim Anggota1     : Sugiri Wiryandono, S.H., M.Hum
Hakim Anggota2     : Doni Prianto, S.H
Panitera           : Sutris Utami, S.H
Penuntut Umum      : Bambang Suparyanto.,S.H
-> File tersimpan: ./OUTPUT/O-putusan_207_pid.sus_2022_pn_blt_20261001203703.txt

DOKUMEN [2/5]: putusan_313_pid.c_2020_pn_blt_20261001203239.txt
Nomor Putusan      : 313/Pid.C/2020/PN Blt
Nama Terdakwa 1    : Suwito
Nama Terdakwa 2    : 
Nama Terdakwa 3    : 
Tuntutan Pidana    : m

Nomor Putusan      : 34/Pid.C/2026/PN Blt
Nama Terdakwa 1    : LANANG TRI ARDIANSYAH
Nama Terdakwa 2    : 
Nama Terdakwa 3    : 
Tuntutan Pidana    : mabuk di
Tuntutan KUHP      : Pasal 316 KUHP
Tuntukan Hukuman   : 
Putusan Pidana     : mabuk di
Putusan Hukuman    : Rp200
Tanggal Putusan    : 24 April 2026
Hakim Ketua        : Taufiq Noor Hayat, S.H
Hakim Anggota1     : 
Hakim Anggota2     : 
Panitera           : Gita Permana Santoso, S.Kom., S.H
Penuntut Umum      : 
-> File tersimpan: ./OUTPUT/O-putusan_34_pid.c_2026_pn_blt_20261001015134.txt

DOKUMEN [5/5]: putusan_62_pid.b_2026_pn_blt_20261001014143.txt
Nomor Putusan      : 62/Pid.B/2026/PN Blt
Nama Terdakwa 1    : Irwanto
Nama Terdakwa 2    : 
Nama Terdakwa 3    : 
Tuntutan Pidana    : pencurian dalam keadaan memberatkan
Tuntutan KUHP      : Pasal 477 ayat (1) huruf g KUHP
Tuntukan Hukuman   : 3 (Tiga) Tahun dan 6 (Enam) Bulan3 (tiga) Tahun dan 6 (enam) 3
Putusan Pidana     : pencurian dalam keadaan yang
Putusan Hukuman    : 3 (t

## 6. Verifikasi & Tampilan Tabel DataFrame Hasil Ekstraksi

Di bawah ini disajikan tabel DataFrame dari file `courtHistory.csv` menggunakan `pandas` untuk memverifikasi kelengkapan seluruh atribut entitas dari ke-5 dokumen putusan perkara Pengadilan Negeri Blitar.

In [6]:
# Baca file CSV hasil ekstraksi menggunakan pandas
df_hasil = pd.read_csv("courtHistory.csv")

# Atur opsi tampilan pandas agar kolom terbaca dengan jelas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Tampilkan 5 baris data hasil ekstraksi
df_hasil

,Nomor Putusan,Nama Terdakwa 1,Nama Terdakwa 2,Nama Terdakwa 3,Tuntutan Pidana,Tuntutan KUHP,Tuntukan Hukuman,Putusan Pidana,Putusan Hukuman,Tanggal Putusan,Hakim Ketua,Hakim Anggota1,Hakim Anggota2,Panitera,Penuntut Umum
0,207/Pid.Sus/2022/PN Blt,Kateni Bin Alm Jaimin,NaN,NaN,menganiaya dan/atau,Pasal 91 B ayat (1) jo Pasal 66 A ayat (1) UU RI No,4 (empat) Bulan dan,PENGANIAYAAN,4 (empat) Bulan dan denda sebesar Rp 3,29 Agustus 2022,"Dr. Wisnu Widiastuti, S.H.,, M.H.um","Sugiri Wiryandono, S.H., M.Hum","Doni Prianto, S.H","Sutris Utami, S.H","Bambang Suparyanto.,S.H"
1,313/Pid.C/2020/PN Blt,Suwito,NaN,NaN,mabuk ditempat umum dan mengganggu,Pasal 492 Ayat 1 KUHP,NaN,mabuk ditempat umum dan mengganggu,5 (lima) hari,14 Pebruari 2020,"SUCI ASTRI PRAMAWATI, S.H., M.HUM",NaN,NaN,"SUTRIS UTAMI, S.H",NaN
2,32/Pid.B/2026/PN Blt,Sholikhin Alias Likok Bin Ginar,NaN,NaN,melakukan penganiayaan,NaN,10 (sepuluh) bulan,melakukan,10 (sepuluh) bulan,16 April 2026,"Agus Darmanto, S.H., M.H","Raden Rajendra Mohni Iswoyokusumo, S.H., M.H","Abraham Yoseph Titapasanea, S.H., M.H","H. Mukhayani, S.H., M.H","Ainur Rofiq, S.H"
3,34/Pid.C/2026/PN Blt,LANANG TRI ARDIANSYAH,NaN,NaN,mabuk di,Pasal 316 KUHP,NaN,mabuk di,Rp200,24 April 2026,"Taufiq Noor Hayat, S.H",NaN,NaN,"Gita Permana Santoso, S.Kom., S.H",NaN
4,62/Pid.B/2026/PN Blt,Irwanto,NaN,NaN,pencurian dalam keadaan memberatkan,Pasal 477 ayat (1) huruf g KUHP,3 (Tiga) Tahun dan 6 (Enam) Bulan3 (tiga) Tahun dan 6 (enam) 3,pencurian dalam keadaan yang,3 (tiga) tahun,15 April 2026,"Agus Darmanto, S.H., M.H","Raden Rajendra Mohni Iswoyokusumo, S.H., M.H","Abraham Yoseph Titapasanea, S.H., M.H","Sukri Safar, S.H","Dwi Budi Setiari, S.H., M.H"


## 7. Kesimpulan dan Analisis Hasil

Berdasarkan hasil uji coba Rule-Based Information Extraction pada dokumen putusan Pengadilan Negeri Blitar:
1. **Tingkat Akurasi Pola**: Pola Regular Expression (`re`) yang dirancang berhasil mengekstrak entitas penting seperti **Nomor Putusan**, **Nama Terdakwa**, **Amar Vonis/Hukuman**, **Tanggal Putusan**, dan **Aparatur Pengadilan (Hakim & Panitera)** secara konsisten.
2. **Karakteristik Dokumen Putusan**: Format dokumen putusan pidana biasa (*Pid.B*) dan pidana khusus (*Pid.Sus*) memiliki kelengkapan entitas penuh (Majelis Hakim lengkap dengan 3 hakim, Jaksa Penuntut Umum, serta berkas tuntutan dan putusan). Sementara pada perkara tindak pidana cepat/ringan (*Pid.C*), persidangan dipimpin oleh Hakim Tunggal dengan format catatan cepat.
3. **Keunggulan Sistem Rule-Based**: Pendekatan rule-based sangat andal untuk dokumen hukum yang memiliki struktur formal berpola tetap (*domain-specific text*), dengan kecepatan komputasi yang sangat cepat tanpa memerlukan inferensi model AI yang berat.